In [14]:
import heapq

def greedy_bfs(graph, heuristics, start, goal):
    pq = [(heuristics[start], start, [start])]
    visited = set()

    while pq:
        h, current_node, path = heapq.heappop(pq)

        if current_node == goal:
            return path

        if current_node not in visited:
            visited.add(current_node)
            
            for n in graph[current_node]:
                if n not in visited:
                    heapq.heappush(pq, (heuristics[n], n, path + [n]))
    
    return None

graph = {
    'A': ['B', 'C', 'D'],
    'B': ['E'],
    'C': ['F'],
    'D': ['F'],
    'E': ['G'],
    'F': ['G']
}

heuristics = {
    'A': 40, 'B': 32, 'C': 25, 'D': 35, 
    'E': 19, 'F': 17, 'G': 0
}

path = greedy_bfs(graph, heuristics, 'A', 'G')
print(f"Path from A to G: {' -> '.join(path) if path else 'No path found'}")


Path from A to G: A -> C -> F -> G


In [15]:

import heapq

def A_star(graph, heuristic, start, goal):
    pq = [(heuristic[start], start, [start], 0)]
    visited = set()
    nodes_expanded = 0

    while pq:
        _, current_node, path, actual_cost = heapq.heappop(pq)
        
        if current_node in visited:
            continue
        
        visited.add(current_node)
        nodes_expanded += 1

        if current_node == goal:
            return path, actual_cost, nodes_expanded

        for n, weight in graph[current_node]:
            if n not in visited:
                new_cost = actual_cost + weight
                new_f_score = new_cost + heuristic[n]
                new_path = path + [n]
                heapq.heappush(pq, (new_f_score, n, new_path, new_cost))

    return None, None, nodes_expanded

graph = {
    'A': [('B', 1), ('C', 4)],
    'B': [('D', 2), ('E', 5)],
    'C': [('F', 1)],
    'D': [('G', 1)],
    'E': [('G', 2)],
    'F': [('G', 3)],
    'G': []
}

heuristic = {
    'A': 7,
    'B': 6,
    'C': 2,
    'D': 1,
    'E': 4,
    'F': 2,
    'G': 0
}

path_found, total_cost_result, expanded_nodes_count = A_star(graph, heuristic, 'A', 'G')

if path_found:
    print(f"Path found: {path_found}")
    print(f"Total actual cost: {total_cost_result}")
    print(f"Number of Nodes expanded: {expanded_nodes_count}")
else:
    print("No path found.")
    print(f"Number of Nodes expanded: {expanded_nodes_count}")



Path found: ['A', 'B', 'D', 'G']
Total actual cost: 4
Number of Nodes expanded: 5


In [16]:
import heapq
from copy import deepcopy

GOAL = [[1, 2, 3],
        [4, 5, 6],
        [7, 8, 0]]

GOAL_POSITIONS = {GOAL[r][c]: (r, c) for r in range(3) for c in range(3)}

MOVES = [(-1, 0, "UP"), (1, 0, "DOWN"), (0, -1, "LEFT"), (0, 1, "RIGHT")]


def misplaced_tiles(state):
    return sum(
        1 for r in range(3) for c in range(3)
        if state[r][c] != 0 and state[r][c] != GOAL[r][c]
    )


def manhattan_distance(state):
    total = 0
    for r in range(3):
        for c in range(3):
            tile = state[r][c]
            if tile != 0:
                gr, gc = GOAL_POSITIONS[tile]
                total += abs(r - gr) + abs(c - gc)
    return total


def find_blank(state):
    return next((r, c) for r in range(3) for c in range(3) if state[r][c] == 0)


def get_neighbors(state):
    br, bc = find_blank(state)
    neighbors = []
    for dr, dc, direction in MOVES:
        nr, nc = br + dr, bc + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = deepcopy(state)
            new_state[br][bc], new_state[nr][nc] = new_state[nr][nc], new_state[br][bc]
            neighbors.append((new_state, direction))
    return neighbors


def state_to_tuple(state):
    return tuple(cell for row in state for cell in row)


def a_star(start, heuristic):
    heap = [(heuristic(start), 0, start, [])]
    visited = set()
    nodes_explored = 0

    while heap:
        f, g, state, path = heapq.heappop(heap)
        s_tuple = state_to_tuple(state)
        if s_tuple in visited:
            continue
        visited.add(s_tuple)
        nodes_explored += 1

        if state == GOAL:
            return path, nodes_explored, g

        for neighbor, direction in get_neighbors(state):
            if state_to_tuple(neighbor) not in visited:
                g_new = g + 1
                f_new = g_new + heuristic(neighbor)
                heapq.heappush(heap, (f_new, g_new, neighbor, path + [direction]))

    return None, nodes_explored, -1


def print_board(state, label=""):
    if label:
        print(f"\n  {label}")
    print("  ┌───────┐")
    for row in state:
        cells = " ".join(str(t) if t != 0 else "·" for t in row)
        print(f"  │ {cells} │")
    print("  └───────┘")


def compare(start):
    print_board(start, "Start")
    print_board(GOAL, "Goal")

    for name, h_fn in [("Misplaced Tiles", misplaced_tiles),
                       ("Manhattan Distance", manhattan_distance)]:
        path, explored, cost = a_star(deepcopy(start), h_fn)
        print(f"\n[{name}]")
        print(f"  Moves    : {cost}")
        print(f"  Nodes    : {explored}")
        print(f"  Path     : {' → '.join(path)}")

hard = [[8, 6, 7],
        [2, 5, 4],
        [3, 0, 1]]

compare(hard)


  Start
  ┌───────┐
  │ 8 6 7 │
  │ 2 5 4 │
  │ 3 · 1 │
  └───────┘

  Goal
  ┌───────┐
  │ 1 2 3 │
  │ 4 5 6 │
  │ 7 8 · │
  └───────┘

[Misplaced Tiles]
  Moves    : 31
  Nodes    : 143849
  Path     : LEFT → UP → RIGHT → DOWN → RIGHT → UP → LEFT → UP → RIGHT → DOWN → LEFT → LEFT → DOWN → RIGHT → RIGHT → UP → LEFT → UP → LEFT → DOWN → DOWN → RIGHT → UP → UP → LEFT → DOWN → DOWN → RIGHT → UP → RIGHT → DOWN

[Manhattan Distance]
  Moves    : 31
  Nodes    : 21198
  Path     : LEFT → UP → RIGHT → DOWN → RIGHT → UP → LEFT → UP → RIGHT → DOWN → LEFT → LEFT → DOWN → RIGHT → RIGHT → UP → LEFT → UP → LEFT → DOWN → DOWN → RIGHT → UP → UP → LEFT → DOWN → DOWN → RIGHT → UP → RIGHT → DOWN


In [17]:
class TicTacToe:
    def __init__(self):
        self.state = [' '] * 9

    def get_moves(self, state):
        return [i for i, cell in enumerate(state) if cell == ' ']

    def get_utility(self, state):
        wins = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
        for a, b, c in wins:
            if state[a] == state[b] == state[c] != ' ':
                return 1 if state[a] == 'X' else -1
        return 0

    def is_terminal(self, state):
        return self.get_utility(state) != 0 or ' ' not in state

    def result(self, state, move, player):
        new_state = list(state)
        new_state[move] = player
        return new_state


In [18]:
import math

class Node:
    def __init__(self, state, player, move=None):
        self.state = state
        self.player = player
        self.move = move
        self.value = None

class MinimaxAI:
    def get_utility(self, state):
        wins = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
        for a, b, c in wins:
            if state[a] == state[b] == state[c] != ' ':
                return 1 if state[a] == 'X' else -1
        return 0

    def is_terminal(self, state):
        return self.get_utility(state) != 0 or ' ' not in state

    def minimax(self, node):
        if self.is_terminal(node.state):
            node.value = self.get_utility(node.state)
            return node.value

        values = []
        next_p = 'O' if node.player == 'X' else 'X'
        moves = [i for i, cell in enumerate(node.state) if cell == ' ']

        for m in moves:
            new_state = list(node.state)
            new_state[m] = node.player
            child = Node(new_state, next_p, m)
            values.append(self.minimax(child))

        node.value = max(values) if node.player == 'X' else min(values)
        return node.value

    def get_best_move(self, current_state):
        best_val = -math.inf
        best_move = None
        print("Moves:")
        
        for m in [i for i, cell in enumerate(current_state) if cell == ' ']:
            new_state = list(current_state)
            new_state[m] = 'X'
            child = Node(new_state, 'O', m)
            val = self.minimax(child)
            print(f"Index {m}: {val}")
            
            if val > best_val:
                best_val = val
                best_move = m
        
        print(f"Selected: {best_move}")
        return best_move

ai = MinimaxAI()
board = ['X', 'O', ' ', 
         ' ', 'O', ' ', 
         ' ', ' ', 'X']
ai.get_best_move(board)

Moves:
Index 2: -1
Index 3: -1
Index 5: -1
Index 6: -1
Index 7: 0
Selected: 7


7